# Exercise 1.4: Exploring open data

From *Programming in High Energy Particle Physics*, Chapter 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch01/ex01_4_solution.ipynb)

Exercise 1.4 Exploring open data. Inspect the CMS [DoubleMuon Run2016G NanoAODv9 dataset](https://opendata.cern.ch/record/30522) using uproot. The selected ROOT file is 66,090,536 bytes. Find the `Events` TTree, list branch names and types, and estimate stored bytes per event by dividing the portal-reported file size by entries. HTTPS range reads fall back to a clearly labelled synthetic file only if it fails.

<details><summary>Hint</summary>

Call `tree.show()` or inspect `tree.keys()` and `tree[name].typename`. The file size includes metadata and compression; this ratio is only an estimate.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot numpy")

### Load the public CMS ROOT file

In [ ]:
import numpy as np, uproot
from pathlib import Path
import tempfile, urllib.request

DATA_URL = "https://opendata.cern.ch/eos/opendata/cms/Run2016G/DoubleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v2/2430000/4E3A26DE-E53B-A844-8048-36376617AE8D.root"
path = DATA_URL
file_size = 66090536  # bytes, from CERN Open Data record 30522
source = "CMS Run2016G DoubleMuon NanoAODv9"
try:
    # uproot fetches only the HTTPS byte ranges needed for the tree.
    with uproot.open(path, timeout=45) as root_file:
        assert "Events" in root_file
except (OSError, TimeoutError, urllib.error.URLError) as exc:
    print(f"CMS download failed ({exc}); using synthetic fallback, not CERN data.")
    source = "synthetic fallback"
    path = Path(tempfile.gettempdir()) / "exercise_1_4_synthetic.root"
    rng = np.random.default_rng(42)
    with uproot.recreate(path) as root_file:
        root_file["Events"] = {"run": np.full(500, 1, dtype=np.uint32),
                               "luminosityBlock": np.ones(500, dtype=np.uint32),
                               "Muon_pt": rng.exponential(35, 500).astype(np.float32),
                               "Muon_eta": rng.normal(0, 1, 500).astype(np.float32)}
    file_size = path.stat().st_size
print("Source:", source)


In [ ]:
with uproot.open(path) as root_file:
    print("TTrees:", root_file.keys())
    tree = root_file["Events"]
    branch_names = list(tree.keys())
    branch_types = {name: tree[name].typename for name in ["run", "luminosityBlock", "Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge"] if name in tree}
    entries = tree.num_entries
bytes_per_event = file_size / entries
print("Branches (first 12):", branch_names[:12])
print("Selected branch types:", branch_types)
print(f"{entries} events, {file_size} bytes, {bytes_per_event:.1f} stored bytes/event")
assert entries > 0 and bytes_per_event > 0

### Interpretation

For the CMS file, `Events` contains `run`, `luminosityBlock`, and jagged muon branches such as `Muon_pt`, `Muon_eta`, `Muon_phi`, and `Muon_charge`. File bytes divided by entries includes metadata and compression, so it approximates stored bytes per event. If the download failed, the printed source identifies a synthetic fallback; its result is not a CMS measurement.